# 第2章 计息惯例、现金流与货币时间价值 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch02_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch02_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 6：房贷计算器（等额本息月供 + 逐月本金/利息明细 + 构成图）


In [ ]:
import numpy as np
from fi import cashflow as cf, plotting
plotting.use_chinese_style()
P, rate, k, N = 1_000_000, 0.05, 12, 360
i = rate / k
pmt = cf.annuity_payment(P, rate, N, freq=k)
print(f'等额本息月供 = {pmt:,.2f} 元   总还款 = {pmt*N:,.2f}   总利息 = {pmt*N-P:,.2f}')
bal, prin, intr = P, [], []
for _ in range(N):
    it = bal * i; pr = pmt - it
    intr.append(it); prin.append(pr); bal -= pr
m = np.arange(1, N+1)
fig, ax = plotting.new_axes()
ax.stackplot(m, prin, intr, labels=['本金', '利息'])
ax.set_xlabel('月'); ax.set_ylabel('月供构成（元）'); ax.set_title('等额本息：前期几乎全是利息'); ax.legend(loc='upper right')
fig.tight_layout()
import pandas as pd
print(pd.DataFrame({'月':[1,12,120,360],'本金':[round(prin[j-1],2) for j in (1,12,120,360)],'利息':[round(intr[j-1],2) for j in (1,12,120,360)]}).to_string(index=False))


## 编程实验 7：year_fraction 跨闰年比较（ACT/ACT vs ACT/365）


In [ ]:
import datetime as dt
s, e = dt.date(2026,3,15), dt.date(2026,9,15)   # 非闰年内：二者相同
print('2026-03-15→09-15  ACT/ACT=%.6f  ACT/365=%.6f' % (cf.year_fraction(s,e,'ACT/ACT'), cf.year_fraction(s,e,'ACT/365')))
s2, e2 = dt.date(2027,12,15), dt.date(2028,6,15)  # 跨入闰年 2028：分叉
print('2027-12-15→2028-06-15  ACT/ACT=%.6f  ACT/365=%.6f' % (cf.year_fraction(s2,e2,'ACT/ACT'), cf.year_fraction(s2,e2,'ACT/365')))
print('解释：区间跨入闰年 2028 后，ACT/ACT 对 2028 部分按 366 折算，与 ACT/365 出现千分位差异')


## 编程实验 8：多惯例应计利息函数 + QuantLib 对拍


In [ ]:
import QuantLib as ql
settle, prev, nxt = dt.date(2026,6,15), dt.date(2026,3,15), dt.date(2026,9,15)
for conv in ['ACT/ACT','30/360','ACT/365']:
    ai = cf.accrued_interest(settle, prev, nxt, 0.03, freq=2, face=100, convention=conv)
    print(f'fi 应计({conv}) = {ai:.6f}')
ql.Settings.instance().evaluationDate = ql.Date(15,6,2026)
sched = ql.Schedule(ql.Date(15,3,2026), ql.Date(15,3,2029), ql.Period(ql.Semiannual),
                    ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Backward, False)
bond = ql.FixedRateBond(0, 100.0, sched, [0.03], ql.ActualActual(ql.ActualActual.ISMA))
ai_ql = bond.accruedAmount(ql.Date(15,6,2026))
ai_fi = cf.accrued_interest(settle, prev, nxt, 0.03, freq=2, face=100, convention='ACT/ACT')
print(f'对拍 ACT/ACT: fi={ai_fi:.6f}  QuantLib={ai_ql:.6f}  误差={abs(ai_fi-ai_ql):.2e}')
